In [1]:
"""
Create an income raster for Nigeria using OnStove income_estimation.
This script uses only method 1 (AWE estimation) in this notebook.
Method 2 (Income CSV interpolation) is intentionally commented out.
"""

import os
import shutil
from onstove import OnStove

# User parameters (paths relative to this notebook folder)
model_pickle_path = os.path.normpath("../../example/NGA/model_inputs.pkl")
scenario_csv_path = os.path.normpath("../../example/NGA/soc_specs.csv")
output_directory = "dataset_big"

# Method 2 not used in this notebook (kept as reference)
# use_income_csv = True
# income_csv_path = "dataset_big/nigeria_income_percentiles.csv"

# Required for AWE mode
gini_value = 0.339      # Example: 0.35
gdp_pc_value = 2139    # Example: 2200.0
pareto_weight = 0.32

# Output raster column and name (AWE writes absolute_wealth)
output_variable = "absolute_wealth"
output_raster_name = "income_nigeria"

# Basic input checks
if not os.path.exists(model_pickle_path):
    raise FileNotFoundError(f"Model pickle not found: {model_pickle_path}")
if not os.path.exists(scenario_csv_path):
    raise FileNotFoundError(f"Scenario csv not found: {scenario_csv_path}")
os.makedirs(output_directory, exist_ok=True)

# 1) Load prepared model and scenario
model = OnStove.read_model(model_pickle_path)
model.read_scenario_data(scenario_csv_path, delimiter=",")
model.output_directory = output_directory

# 2) Configure inputs for selected income mode
# AWE only branch
if gini_value is None or gdp_pc_value is None:
    raise ValueError("Set gini_value and gdp_pc_value for AWE mode")
model.specs["gini"] = float(gini_value)
model.specs["gdp_pc"] = float(gdp_pc_value)
model.income_estimation(awe=True, income_data=None, pareto_weight=pareto_weight)

# 3) Quick checks
if "relative_wealth" not in model.gdf.columns:
    raise KeyError("Model gdf is missing 'relative_wealth', required by income_estimation")
if output_variable not in model.gdf.columns:
    raise KeyError(f"Expected output column '{output_variable}' not found in model.gdf")

# 4) Export raster (suppresses OnStove's internal print statements)
import contextlib
import io
with contextlib.redirect_stdout(io.StringIO()):
    model.to_raster(variable=output_variable)

# Move file to final destination and clean up
src_name = f"{output_variable}_mean.tif"
src_path = os.path.join(output_directory, "Rasters", src_name)
dst_path = os.path.join(output_directory, f"{output_raster_name}.tif")
if os.path.exists(src_path):
    if os.path.exists(dst_path):
        os.remove(dst_path)
    os.replace(src_path, dst_path)
    print(f"Raster moved to: {dst_path}")

# Delete the temporary Rasters directory
rasters_dir = os.path.join(output_directory, "Rasters")
if os.path.exists(rasters_dir):
    shutil.rmtree(rasters_dir)
    print(f"Temporary directory deleted: {rasters_dir}")

print("Income mode: AWE")
print(f"Final raster: {dst_path if os.path.exists(dst_path) else 'NOT FOUND'}")


Raster moved to: dataset_big\income_nigeria.tif
Temporary directory deleted: dataset_big\Rasters
Income mode: AWE
Final raster: dataset_big\income_nigeria.tif


In [2]:
"""
Reproject income raster to EPSG:3857 (Web Mercator), normalize, and create vehicle_possibility.
Vehicle possibility = normalized income with exponent applied.
"""

import rioxarray
import numpy as np
import xarray as xr
from rasterio.warp import Resampling

def main():
    
    # User parameters
    input_path = "dataset_big/income_nigeria.tif"
    output_vehicle_path = "dataset_big/vehicle_possibility.tif"
    target_crs = "EPSG:3857"
    target_resolution = 1000  # meters in EPSG:3857 (1 km)
    output_nodata = -9999.0

    # Shape parameter for vehicle possibility (applied on normalized income).
    # 1.0 keeps linear relation, >1 emphasizes high-income pixels, <1 flattens differences.
    income_exponent = 1.0
    
    # Read the income raster in its original CRS (already on Nigeria from OnStove)
    print(f"Reading income raster from {input_path}...")
    income_da = rioxarray.open_rasterio(input_path)
    original_crs = income_da.rio.crs
    original_nodata = income_da.rio.nodata
    print(f"Original CRS: {original_crs}")
    print(f"Original resolution: {income_da.rio.resolution()}")
    print(f"Original nodata value: {original_nodata}")
    print(f"Array contains NaN values: {np.isnan(income_da.values).any()}")
    
    # Reproject to target CRS
    print(f"Reprojecting to {target_crs} with resolution {target_resolution} m...")
    income_3857 = income_da.rio.reproject(
        target_crs,
        resolution=target_resolution,
        resampling=Resampling.nearest
    )
    income_3857 = income_3857.fillna(output_nodata)
    income_3857.rio.write_nodata(output_nodata, inplace=True)
    
    # Normalize income to [0, 1] and apply exponent to create vehicle_possibility.
    print("Normalizing income to [0, 1] and applying exponent...")
    data = income_3857.values.astype(np.float32, copy=True)
    valid = (np.isfinite(data)) & (data != output_nodata)
    if original_nodata is not None:
        valid = valid & (data != original_nodata)
    
    if valid.any():
        vmin = data[valid].min()
        vmax = data[valid].max()
        
        if vmax > vmin:
            data[valid] = (data[valid] - vmin) / (vmax - vmin)
        else:
            data[valid] = 0.0
        
        # Apply exponent on already normalized data
        data[valid] = np.clip(data[valid], 0.0, 1.0)
        data[valid] = np.power(data[valid], income_exponent, dtype=np.float32)
        
        data[~valid] = output_nodata
        
        print(f"Data normalized and exponent applied: min={float(data[valid].min()):.4f}, max={float(data[valid].max()):.4f}")
        print(f"Nodata value={output_nodata}, nodata pixels={int((~valid).sum())}")
    else:
        raise ValueError("No valid pixels found for normalization.")
    
    # Build vehicle_possibility raster (single output)
    vehicle_possibility = xr.DataArray(
        data,
        coords=income_3857.coords,
        dims=income_3857.dims,
        attrs=income_3857.attrs,
        name="vehicle_possibility",
    )
    vehicle_possibility.rio.write_crs(target_crs, inplace=True)
    vehicle_possibility.rio.write_nodata(output_nodata, inplace=True)
    vehicle_possibility.attrs["source"] = "income_only"
    vehicle_possibility.attrs["income_exponent"] = float(income_exponent)

    print(f"Writing vehicle possibility raster to {output_vehicle_path}...")
    vehicle_possibility.rio.to_raster(output_vehicle_path, compress="DEFLATE", nodata=output_nodata)

    out = vehicle_possibility.values
    out_valid = np.isfinite(out) & (out != output_nodata)
    print(f"Done. Vehicle possibility raster saved as {output_vehicle_path} in {target_crs}")
    print(f"Approach: income normalized with exponent={income_exponent}")
    print(f"Output range: min={float(out[out_valid].min()):.4f}, max={float(out[out_valid].max()):.4f}")
    
if __name__ == "__main__":
    main()



Reading income raster from dataset_big/income_nigeria.tif...
Original CRS: EPSG:3395
Original resolution: (1000.0, -1000.0)
Original nodata value: 0.0
Array contains NaN values: False
Reprojecting to EPSG:3857 with resolution 1000 m...
Normalizing income to [0, 1] and applying exponent...
Data normalized and exponent applied: min=0.0000, max=1.0000
Nodata value=-9999.0, nodata pixels=802386
Writing vehicle possibility raster to dataset_big/vehicle_possibility.tif...
Done. Vehicle possibility raster saved as dataset_big/vehicle_possibility.tif in EPSG:3857
Approach: income normalized with exponent=1.0
Output range: min=0.0000, max=1.0000


In [3]:
"""
Vehicle allocation rationale

This cell distributes a fixed total number of vehicles across pixels in Nigeria,
combining:
1) vehicle_possibility (derived from income only)
2) population per pixel.

Method idea:
- the pixel with the highest vehicle_possibility is constrained to 100% adoption,
- all other pixels receive a lower adoption share, scaled smoothly,
- a calibration parameter (alpha) is found so that the sum of allocated cars
  is exactly equal to the national target (11,600,000). [OICA, 2020]

Final outputs are two separate rasters:
- vehicles_allocation_share.tif: share (%) of population owning a car in each pixel
- vehicles_allocation_n_effettivo.tif: number of cars allocated to each pixel
"""

import numpy as np
import xarray as xr
import rioxarray
from rasterio.warp import Resampling

# Inputs
vehicle_possibility_path = "dataset_big/vehicle_possibility.tif"
population_path = "dataset_big/Population.tif"
output_share_path = "dataset_big/vehicles_allocation_share.tif"
output_cars_path = "dataset_big/vehicles_allocation_n_effettivo.tif"

total_vehicles = 11_600_000
output_nodata = -9999.0

def total_cars_for_alpha(alpha, s_norm, pop):
    """Return total allocated cars for a given alpha."""
    rates = np.power(s_norm, alpha, dtype=np.float64)
    return float(np.sum(pop * rates, dtype=np.float64))

# 1) Read rasters and align population to the vehicle-possibility grid
score_da = rioxarray.open_rasterio(vehicle_possibility_path)
pop_da = rioxarray.open_rasterio(population_path)
pop_da = pop_da.rio.reproject_match(score_da, resampling=Resampling.nearest)

score_nodata = score_da.rio.nodata
pop_nodata = pop_da.rio.nodata

score = score_da.values.astype(np.float64, copy=False)
pop = pop_da.values.astype(np.float64, copy=False)

# 2) Build valid mask: finite, positive population, and non-nodata values
valid = np.isfinite(score) & np.isfinite(pop) & (pop > 0)
if score_nodata is not None:
    valid &= score != score_nodata
if pop_nodata is not None:
    valid &= pop != pop_nodata

if not valid.any():
    raise ValueError("No valid pixels found after masking.")

# 3) Convert vehicle possibility to [0, 1] among valid pixels
s = score[valid]
s_min = float(np.min(s))
s_max = float(np.max(s))

if s_max <= s_min:
    raise ValueError("Vehicle possibility has no variation on valid pixels.")

s_norm = (s - s_min) / (s_max - s_min)
s_norm = np.clip(s_norm, 0.0, 1.0)

# Force one max pixel to exactly 1.0 => 100% adoption in that pixel
max_idx = int(np.argmax(s))
s_norm[max_idx] = 1.0

pop_valid = pop[valid]

# 4) Calibrate alpha with bisection so sum(cars_i) = total_vehicles
cars_at_alpha0 = total_cars_for_alpha(0.0, s_norm, pop_valid)
if total_vehicles > cars_at_alpha0:
    raise ValueError(
        f"Requested total_vehicles={total_vehicles:,} exceeds population capacity={cars_at_alpha0:,.0f}."
    )

left, right = 0.0, 40.0
for _ in range(80):
    mid = 0.5 * (left + right)
    cars_mid = total_cars_for_alpha(mid, s_norm, pop_valid)
    if cars_mid > total_vehicles:
        left = mid
    else:
        right = mid

alpha = 0.5 * (left + right)

# 5) Compute final per-pixel outputs
rates_valid = np.power(s_norm, alpha, dtype=np.float64)  # 0..1
cars_valid = pop_valid * rates_valid

share_percent = np.full(score.shape, output_nodata, dtype=np.float32)
n_effettivo = np.full(score.shape, output_nodata, dtype=np.float32)

share_percent[valid] = (rates_valid * 100.0).astype(np.float32)
n_effettivo[valid] = cars_valid.astype(np.float32)

# 6) Save to two separate GeoTIFF files (same grid, same CRS)
share_da = xr.DataArray(
    share_percent,
    coords=score_da.coords,
    dims=score_da.dims,
    name="share",
)
share_da.rio.write_crs(score_da.rio.crs, inplace=True)
share_da.rio.write_nodata(output_nodata, inplace=True)
share_da.attrs["long_name"] = "vehicles_allocation_share_percent"
share_da.attrs["total_vehicles_target"] = int(total_vehicles)
share_da.attrs["alpha"] = float(alpha)
share_da.rio.to_raster(output_share_path, compress="DEFLATE", nodata=output_nodata)

cars_da = xr.DataArray(
    n_effettivo,
    coords=score_da.coords,
    dims=score_da.dims,
    name="n_effettivo",
)
cars_da.rio.write_crs(score_da.rio.crs, inplace=True)
cars_da.rio.write_nodata(output_nodata, inplace=True)
cars_da.attrs["long_name"] = "vehicles_allocation_number_of_cars"
cars_da.attrs["total_vehicles_target"] = int(total_vehicles)
cars_da.attrs["alpha"] = float(alpha)
cars_da.rio.to_raster(output_cars_path, compress="DEFLATE", nodata=output_nodata)

# 7) Minimal summary of allocation result
total_allocated = float(np.sum(n_effettivo[n_effettivo != output_nodata], dtype=np.float64))
max_share = float(np.max(share_percent[share_percent != output_nodata]))

print(f"Estimated alpha: {alpha:.6f}")
print(f"Allocated vehicles total: {total_allocated:,.0f}")
print(f"Target vehicles total: {total_vehicles:,.0f}")
print(f"Max share (%): {max_share:.2f}")
print(f"Share raster written: {output_share_path}")
print(f"Cars raster written: {output_cars_path}")

Estimated alpha: 10.808621
Allocated vehicles total: 11,600,000
Target vehicles total: 11,600,000
Max share (%): 100.00
Share raster written: dataset_big/vehicles_allocation_share.tif
Cars raster written: dataset_big/vehicles_allocation_n_effettivo.tif


In [4]:
# Quick stats on car-ownership share distribution per pixel

import numpy as np
import rioxarray

share_path = "dataset_big/vehicles_allocation_share.tif"
share_da = rioxarray.open_rasterio(share_path)

nodata = share_da.rio.nodata
data = share_da.values.astype(np.float64, copy=False)

valid = np.isfinite(data)
if nodata is not None:
    valid &= data != nodata

vals = data[valid]

if vals.size == 0:
    raise ValueError("No valid values found in share raster.")

print("=== SHARE DISTRIBUTION (% auto per pixel) ===")
print(f"Valid cells: {vals.size:,}")
print(f"Min: {vals.min():.2f}%")
print(f"P10: {np.percentile(vals, 10):.2f}%")
print(f"P25: {np.percentile(vals, 25):.2f}%")
print(f"Median: {np.percentile(vals, 50):.2f}%")
print(f"Mean: {vals.mean():.2f}%")
print(f"P75: {np.percentile(vals, 75):.2f}%")
print(f"P90: {np.percentile(vals, 90):.2f}%")
print(f"P95: {np.percentile(vals, 95):.2f}%")
print(f"Max: {vals.max():.2f}%")

# Bins in percentage points
bins = np.array([0, 1, 5, 10, 20, 40, 60, 80, 95, 100.0001], dtype=np.float64)
labels = [
    "0-1%", "1-5%", "5-10%", "10-20%", "20-40%",
    "40-60%", "60-80%", "80-95%", "95-100%"
]

counts, _ = np.histogram(vals, bins=bins)
shares = counts / vals.size * 100.0

print("\n=== CELLS BY SHARE CLASS ===")
for lbl, c, s in zip(labels, counts, shares):
    print(f"{lbl:>8}: {c:>8,} cells ({s:>6.2f}%)")

# Extra: how many cells are near full adoption
above_90 = np.sum(vals >= 90)
above_50 = np.sum(vals >= 50)
print("\n=== QUICK THRESHOLDS ===")
print(f"Cells >= 50%: {above_50:,} ({above_50/vals.size*100:.2f}%)")
print(f"Cells >= 90%: {above_90:,} ({above_90/vals.size*100:.2f}%)")

=== SHARE DISTRIBUTION (% auto per pixel) ===
Valid cells: 560,635
Min: 0.00%
P10: 0.00%
P25: 0.00%
Median: 0.00%
Mean: 0.28%
P75: 0.00%
P90: 0.00%
P95: 0.00%
Max: 100.00%

=== CELLS BY SHARE CLASS ===
    0-1%:  555,772 cells ( 99.13%)
    1-5%:    1,711 cells (  0.31%)
   5-10%:      523 cells (  0.09%)
  10-20%:      487 cells (  0.09%)
  20-40%:      529 cells (  0.09%)
  40-60%:      370 cells (  0.07%)
  60-80%:      353 cells (  0.06%)
  80-95%:      330 cells (  0.06%)
 95-100%:      560 cells (  0.10%)

=== QUICK THRESHOLDS ===
Cells >= 50%: 1,418 (0.25%)
Cells >= 90%: 691 (0.12%)
